# K4 Track4 Day2 — DeepWeeds
Notebook triển khai các phần chính của bài lab. GPU T4 x2, Internet bật để lấy trọng số ImageNet.
Dùng **dữ liệu đã tải sẵn**, không tải lại ảnh. Chọn mọi cấu hình trên val; test chỉ ở chung kết.
Code được nhúng trong notebook và ghi vào `/kaggle/working/day2_lab_code`.
Chạy đầy đủ tự động: backbone, ablation, công thức kết hợp, suy luận, chung kết 3 seed và xuất kết quả.
Gắn Output của Version 4 bằng Add Input để tái sử dụng 5 backbone đã chạy.


In [ ]:
import os, sys, json, subprocess, importlib
from pathlib import Path
os.chdir('/kaggle/working')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'thop', 'openpyxl'], check=True)
CODE_DIR = Path('/kaggle/working/day2_lab_code')
CODE_DIR.mkdir(exist_ok=True)
SOURCES = {'benchmark.py': '"""Forward-only latency with warmup and GPU synchronization."""\nimport copy\nimport time\nimport numpy as np\nimport torch\n\ndef bench(fn, warmup=10, iters=100, sync=None):\n    if warmup < 10 or iters < 50:\n        raise ValueError(\'Use >=10 warmup and >=50 measured iterations\')\n    sync = sync or (lambda: None)\n    for _ in range(warmup):\n        fn()\n    samples = []\n    for _ in range(iters):\n        sync()\n        start = time.perf_counter()\n        fn()\n        sync()\n        samples.append((time.perf_counter() - start) * 1000)\n    p50, p95, p99 = np.percentile(samples, [50, 95, 99])\n    return dict(p50=float(p50), p95=float(p95), p99=float(p99), mean=float(np.mean(samples)), n=iters)\n\ndef latency_report(model, batch_size, img_size, dtype=\'fp32\', device=\'cuda\', warmup=10, iters=100, k_views=1):\n    if dtype not in (\'fp32\', \'amp\', \'fp16\') or k_views < 1:\n        raise ValueError(\'Invalid dtype or view count\')\n    device = torch.device(device)\n    clone = copy.deepcopy(model).to(device).eval()\n    if dtype == \'fp16\':\n        clone.half()\n    else:\n        clone.float()\n    x = torch.randn(batch_size, 3, img_size, img_size, device=device,\n                    dtype=torch.float16 if dtype == \'fp16\' else torch.float32)\n    def forward():\n        with torch.inference_mode(), torch.autocast(device.type, enabled=dtype == \'amp\' and device.type == \'cuda\'):\n            for i in range(k_views):\n                clone(x.flip(-1) if i % 2 else x)\n    sync = (lambda: torch.cuda.synchronize(device)) if device.type == \'cuda\' else None\n    result = bench(forward, warmup, iters, sync)\n    return {**result, \'gpu\': torch.cuda.get_device_name(device) if device.type == \'cuda\' else \'CPU\',\n            \'dtype\': dtype, \'batch\': batch_size, \'img_size\': img_size, \'K\': k_views,\n            \'images_per_s\': batch_size * 1000 / result[\'p50\'], \'torch\': str(torch.__version__),\n            \'preprocessing\': False, \'scope\': \'forward; aggregation excluded\', \'warmup\': warmup}\n\ndef tta_latency(model, k_views, **kw):\n    return latency_report(model, k_views=k_views, **kw)\n', 'dataset.py': '"""DeepWeeds datasets: fixed official splits, reproducible loaders."""\nfrom pathlib import Path\nimport random\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSampler\nfrom torchvision import transforms as T\n\nNUM_CLASSES = 9\nCLASS_NAMES = [\'Chinee Apple\', \'Lantana\', \'Parkinsonia\', \'Parthenium\', \'Prickly Acacia\', \'Rubber Vine\', \'Siam Weed\', \'Snake Weed\', \'Negatives\']\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\ndef load_split(labels_dir, fold=0):\n    if fold not in range(5):\n        raise ValueError(\'fold must be 0..4\')\n    return tuple(pd.read_csv(Path(labels_dir) / f\'{s}_subset{fold}.csv\') for s in (\'train\', \'val\', \'test\'))\n\ndef check_split(train_df, val_df, test_df, images_dir):\n    frames = dict(zip((\'train\', \'val\', \'test\'), (train_df, val_df, test_df)))\n    sets, result = {}, {\'n\': {}, \'per_class\': {}, \'overlap\': {}}\n    for name, df in frames.items():\n        if not {\'Filename\', \'Label\'} <= set(df):\n            raise ValueError(f\'{name}: missing Filename/Label\')\n        if df.Filename.isna().any() or df.Filename.duplicated().any() or not df.Label.isin(range(9)).all():\n            raise ValueError(f\'{name}: invalid filenames or labels\')\n        sets[name] = set(df.Filename)\n        result[\'n\'][name] = len(df)\n        result[\'per_class\'][name] = df.Label.value_counts().reindex(range(9), fill_value=0).to_dict()\n        missing = [f for f in df.Filename if not (Path(images_dir) / f).is_file()]\n        if missing:\n            raise ValueError(f\'{name}: {len(missing)} missing images, e.g. {missing[:3]}\')\n    for a, b in ((\'train\', \'val\'), (\'train\', \'test\'), (\'val\', \'test\')):\n        result[\'overlap\'][f\'{a}/{b}\'] = len(sets[a] & sets[b])\n    if any(result[\'overlap\'].values()) or len(set.union(*sets.values())) != 17509:\n        raise ValueError(\'Splits overlap or their union does not contain 17509 images\')\n    if any(abs(result[\'n\'][s] / 17509 - r) > .01 for s, r in zip(frames, (.6, .2, .2))):\n        raise ValueError(\'Split proportions differ from 60/20/20 by more than 1 percentage point\')\n    print(result)\n    return result\n\ndef build_transforms(train, img_size=224, aug=\'basic\', mean=IMAGENET_MEAN, std=IMAGENET_STD):\n    if train:\n        steps = [T.RandomResizedCrop(img_size), T.RandomHorizontalFlip()]\n        if aug == \'color\':\n            steps.append(T.ColorJitter(.2, .2, .2, .05))\n        elif aug == \'trivial\':\n            steps.append(T.TrivialAugmentWide())\n        elif aug == \'randaug\':\n            steps.append(T.RandAugment(num_ops=2, magnitude=9))\n        elif aug != \'basic\':\n            raise ValueError(f\'Unknown augmentation: {aug}\')\n    else:\n        steps = [T.Resize(round(img_size * 256 / 224)), T.CenterCrop(img_size)]\n    return T.Compose(steps + [T.ToTensor(), T.Normalize(mean, std)])\n\nclass DeepWeedsDataset(Dataset):\n    def __init__(self, df, images_dir, transform=None):\n        self.df, self.images_dir, self.transform = df.reset_index(drop=True), Path(images_dir), transform\n    def __len__(self):\n        return len(self.df)\n    def __getitem__(self, i):\n        row = self.df.iloc[i]\n        with Image.open(self.images_dir / row.Filename) as image:\n            image = image.convert(\'RGB\')\n            if self.transform:\n                image = self.transform(image)\n        return image, int(row.Label), str(row.Filename)\n\ndef seed_worker(worker_id):\n    seed = torch.initial_seed() % 2**32\n    random.seed(seed)\n    np.random.seed(seed)\n\ndef make_loader(df, images_dir, transform, batch_size, train, sampler=None, num_workers=2, seed=0):\n    generator = torch.Generator().manual_seed(seed)\n    weighted = None\n    if sampler is not None:\n        if not train or sampler != \'balanced\':\n            raise ValueError(\'balanced sampler is supported only during training\')\n        counts = df.Label.value_counts()\n        weights = torch.tensor([1 / counts[y] for y in df.Label], dtype=torch.double)\n        weighted = WeightedRandomSampler(weights, len(df), replacement=True, generator=generator)\n    return DataLoader(DeepWeedsDataset(df, images_dir, transform), batch_size=batch_size,\n                      shuffle=train and weighted is None, sampler=weighted, num_workers=num_workers,\n                      pin_memory=torch.cuda.is_available(), drop_last=train and len(df) >= batch_size,\n                      worker_init_fn=seed_worker, generator=generator)\n', 'experiments.py': '"""Reusable validation experiments and gated final inference."""\nfrom dataclasses import replace, fields\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom train import run, run_dir, Config\nfrom model import build_model\nfrom inference import predict_logits, view_hflip, aggregate_views, fit_temperature, apply_temperature, fuse_conv_bn, softmax\nfrom benchmark import latency_report\nfrom eval import compute_metrics, save_predictions\n\nBACKBONES = [\'resnet50\', \'resnext50_32x4d\', \'convnext_tiny\', \'deit_small_patch16_224\', \'efficientnet_b0\']\nTRAINING_RECIPES = {\n    \'T00\': {}, \'T01_scratch\': {\'init\': \'scratch\'}, \'T02_frozen\': {\'init\': \'frozen\'},\n    \'T03_color\': {\'aug\': \'color\'}, \'T04_cutmix\': {\'mix\': \'cutmix\'}, \'T05_mixup\': {\'mix\': \'mixup\'},\n    \'T06_ls\': {\'loss\': \'ls\', \'label_smoothing\': .1}, \'T07_focal\': {\'loss\': \'focal\'},\n    \'T08_weighted\': {\'loss\': \'ce_weighted\'}, \'T09_ema\': {\'ema_decay\': .999},\n}\n\ndef train_suite(base, stage=\'backbones\'):\n    configs = ([replace(base, exp_id=f\'B{i:02}\', backbone=name) for i, name in enumerate(BACKBONES, 1)]\n               if stage == \'backbones\' else [replace(base, exp_id=k, **v) for k, v in TRAINING_RECIPES.items()])\n    if base.save_test_predictions:\n        raise ValueError(\'Validation suites must not evaluate test\')\n    results = []\n    for cfg in configs:\n        summary = run_dir(cfg) / \'summary.json\'\n        if summary.exists():\n            # Resume orchestration only if saved configuration matches this experiment.\n            old = json.loads((run_dir(cfg)/\'config.json\').read_text())\n            from dataclasses import asdict\n            if any(old.get(k) != v for k, v in asdict(cfg).items()):\n                raise ValueError(f\'Existing run has another configuration: {cfg.exp_id}\')\n            results.append(json.loads(summary.read_text()))\n        else:\n            results.append(run(cfg))\n        pd.DataFrame(results).to_csv(f\'{stage}_results.csv\', index=False)\n    return pd.DataFrame(results)\n\ndef load_checkpoint(cfg, device=\'cuda\'):\n    model = build_model(cfg.backbone, pretrained=False, drop_rate=cfg.drop_rate, init=\'scratch\')\n    state = torch.load(run_dir(cfg)/\'best.pt\', map_location=\'cpu\', weights_only=True)\n    model.load_state_dict(state[\'model\'])\n    return model.to(device).eval()\n\ndef restore_config(exp_id, seed=0, out_dir=\'runs\'):\n    """Recover the full recipe instead of guessing from a checkpoint name."""\n    saved = json.loads((Path(out_dir)/exp_id/f\'seed{seed}\'/\'config.json\').read_text())\n    return Config(**{field.name: saved[field.name] for field in fields(Config)})\n\ndef inference_suite(cfg, val_loader, device=\'cuda\'):\n    model = load_checkpoint(cfg, device)\n    names, y, z = predict_logits(model, val_loader, device)\n    flipped_names, flipped_y, flipped = predict_logits(model, val_loader, device, view_hflip)\n    if names != flipped_names or not np.array_equal(y, flipped_y):\n        raise ValueError(\'View filenames/labels differ\')\n    T = fit_temperature(z, y)\n    candidates = {\'I00\': (softmax(z), 1, \'fp32\'),\n                  \'I01_hflip_prob\': (aggregate_views([z, flipped], \'prob\'), 2, \'fp32\'),\n                  \'I02_hflip_logit\': (aggregate_views([z, flipped], \'logit\'), 2, \'fp32\'),\n                  \'I03_temperature\': (apply_temperature(z, T), 1, \'fp32\')}\n    fused = fuse_conv_bn(model)\n    fn, fy, fz = predict_logits(fused, val_loader, device)\n    if fn != names or not np.array_equal(fy, y):\n        raise ValueError(\'Fused model filenames differ\')\n    fusion_error = float(np.max(np.abs(z-fz)))\n    candidates[\'I04_fused\'] = (softmax(fz), 1, \'fp32\')\n    import copy\n    half_model = copy.deepcopy(model).half()\n    hn, hy, hz = predict_logits(half_model, val_loader, device)\n    if hn != names or not np.array_equal(hy, y):\n        raise ValueError(\'FP16 filenames/labels differ\')\n    candidates[\'I05_fp16\'] = (softmax(hz), 1, \'fp16\')\n    del half_model\n    rows = []\n    for exp_id, (p, k, dtype) in candidates.items():\n        metrics = compute_metrics(y, p.argmax(1), p)\n        timings = latency_report(fused if exp_id == \'I04_fused\' else model, 1, cfg.img_size,\n                                 dtype=dtype, device=device, k_views=k)\n        rows.append({\'exp_id\': exp_id, \'checkpoint\': str(run_dir(cfg)/\'best.pt\'),\n                     **{key: metrics[key] for key in (\'macro_f1\', \'top1\', \'ece\')}, **timings,\n                     \'temperature\': T if exp_id == \'I03_temperature\' else 1., \'fusion_max_abs\': fusion_error})\n        save_predictions(Path(cfg.pred_dir)/f\'{exp_id}_seed{cfg.seed}_val.csv\', names, y, p)\n    result = pd.DataFrame(rows)\n    result.to_csv(\'inference_results.csv\', index=False)\n    return result\n\ndef final_inference(cfg, val_loader, test_loader, method=\'identity\', calibrated=True, device=\'cuda\'):\n    """Call only after locking the recipe on val. One test pass per view and seed."""\n    if method not in (\'identity\', \'hflip_prob\', \'hflip_logit\'):\n        raise ValueError(\'Unknown final method\')\n    path = Path(cfg.pred_dir)/f\'{cfg.exp_id}_seed{cfg.seed}_test.csv\'\n    if path.exists():\n        raise FileExistsError(\'Final predictions already exist; do not repeat test selection\')\n    model = load_checkpoint(cfg, device)\n    def collect(loader):\n        names, y, z = predict_logits(model, loader, device)\n        if method == \'identity\':\n            return names, y, z\n        ns, ys, flip = predict_logits(model, loader, device, view_hflip)\n        if names != ns or not np.array_equal(y, ys):\n            raise ValueError(\'Mismatched TTA order\')\n        # Prob-space aggregation represented as log probabilities for scalar calibration.\n        return names, y, (z+flip)/2 if method == \'hflip_logit\' else np.log(aggregate_views([z, flip], \'prob\').clip(1e-12))\n    vn, vy, vz = collect(val_loader)\n    T = fit_temperature(vz, vy) if calibrated else 1.\n    names, y, z = collect(test_loader)\n    p = apply_temperature(z, T)\n    save_predictions(path, names, y, p)\n    save_predictions(Path(cfg.pred_dir)/f\'{cfg.exp_id}uncal_seed{cfg.seed}_test.csv\', names, y, softmax(z))\n    save_predictions(Path(cfg.pred_dir)/f\'{cfg.exp_id}_seed{cfg.seed}_val.csv\', vn, vy, apply_temperature(vz, T))\n    result = {key: value for key, value in compute_metrics(y, p.argmax(1), p).items()\n              if key in (\'top1\', \'macro_f1\', \'ece\', \'balanced_acc\')}\n    result.update(exp_id=cfg.exp_id, seed=cfg.seed, temperature=T, method=method)\n    return result\n\ndef export_results(path=\'results.xlsx\'):\n    # Only real measured results are exported; empty sheets never invent scores.\n    mapping = {\'Backbones\': \'backbones_results.csv\', \'Training\': \'training_results.csv\',\n               \'Inference\': \'inference_results.csv\', \'Final\': \'final_results.csv\',\n               \'PerClass\': \'per_class_results.csv\', \'Latency\': \'latency_results.csv\'}\n    tables = {name: pd.read_csv(file) if Path(file).exists() else pd.DataFrame() for name, file in mapping.items()}\n    sources = [df.assign(stage=name) for name, df in tables.items() if \'macro_f1\' in df and name != \'Final\']\n    summary = pd.concat(sources, ignore_index=True) if sources else pd.DataFrame()\n    tables[\'Summary\'] = summary.sort_values(\'macro_f1\', ascending=False).head(10) if len(summary) else pd.DataFrame()\n    with pd.ExcelWriter(path, engine=\'openpyxl\') as writer:\n        for name, table in tables.items():\n            table.to_excel(writer, sheet_name=name, index=False)\n            ws = writer.sheets[name]\n            ws.freeze_panes = \'A2\'\n            if len(table):\n                ws.auto_filter.ref = ws.dimensions\n            for column in ws.columns:\n                ws.column_dimensions[column[0].column_letter].width = min(45, max(14, max(len(str(c.value or \'\')) for c in column)+2))\n    return path\n', 'full_lab.py': '"""Run all required stages, making every selection on validation only."""\nimport json\nimport shutil\nfrom dataclasses import asdict, replace\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom train import run, run_dir\nfrom experiments import (train_suite, restore_config, TRAINING_RECIPES,\n                         inference_suite, final_inference, export_results)\nfrom dataset import make_loader, build_transforms\nfrom eval import main as eval_main, compute_metrics\n\n\ndef import_previous_output(base):\n    """Only import backbone runs; never import final test files from another run."""\n    sources = sorted(Path(\'/kaggle/input\').rglob(\'backbones_results.csv\'))\n    if len(sources) > 1:\n        raise ValueError(\'Attach only one previous DeepWeeds notebook output\')\n    if not sources:\n        print(\'No previous output attached: backbone suite will train from scratch.\', flush=True)\n        return\n    source = sources[0].parent\n    for exp in range(1, 6):\n        relative = Path(\'runs\') / f\'B{exp:02}\' / \'seed0\'\n        if not (source / relative / \'summary.json\').exists():\n            raise ValueError(f\'Incomplete previous backbone: {relative}\')\n        if not relative.exists():\n            shutil.copytree(source / relative, relative)\n            config_path = relative / \'config.json\'\n            metadata = json.loads(config_path.read_text())\n            metadata[\'imported_from\'] = str(source / relative)\n            metadata[\'original_data_paths\'] = {key: metadata[key] for key in (\'images_dir\', \'labels_dir\')}\n            metadata.update(images_dir=base.images_dir, labels_dir=base.labels_dir)\n            config_path.write_text(json.dumps(metadata, indent=2))\n    for curve in (source / \'curves\').glob(\'B*.png\'):\n        Path(\'curves\').mkdir(exist_ok=True)\n        if not (Path(\'curves\') / curve.name).exists():\n            shutil.copy2(curve, Path(\'curves\') / curve.name)\n    print(\'Imported previous five backbone checkpoints:\', source, flush=True)\n\n\ndef reuse_run(source_cfg, target_cfg):\n    """Reuse an identical seed/recipe with explicit provenance."""\n    a, b = asdict(source_cfg), asdict(target_cfg)\n    for key in a:\n        if key != \'exp_id\' and a[key] != b[key]:\n            raise ValueError(f\'Cannot reuse a different recipe: {key}\')\n    target = run_dir(target_cfg)\n    if target.exists():\n        return\n    shutil.copytree(run_dir(source_cfg), target)\n    meta = json.loads((target / \'config.json\').read_text())\n    meta.update(asdict(target_cfg), reused_from=str(run_dir(source_cfg)))\n    (target / \'config.json\').write_text(json.dumps(meta, indent=2))\n    summary = json.loads((target / \'summary.json\').read_text())\n    summary.update(exp_id=target_cfg.exp_id, checkpoint=str(target / \'best.pt\'),\n                   reused_from=str(run_dir(source_cfg)))\n    (target / \'summary.json\').write_text(json.dumps(summary, indent=2))\n    from train import plot_curves\n    history = pd.read_csv(target / \'history.csv\').to_dict(\'records\')\n    plot_curves(history, Path(target_cfg.curves_dir) / f\'{target_cfg.exp_id}_seed{target_cfg.seed}.png\',\n                f\'{target_cfg.exp_id} / reused identical seed and recipe\')\n\n\ndef loaders(cfg, frames):\n    data = json.loads((run_dir(cfg) / \'config.json\').read_text())[\'data_cfg\']\n    transform = build_transforms(False, cfg.img_size, mean=data[\'mean\'], std=data[\'std\'])\n    return [make_loader(df, cfg.images_dir, transform, cfg.batch_size, False,\n                        num_workers=cfg.num_workers) for df in frames]\n\n\ndef run_all(base, val_df, test_df):\n    import_previous_output(base)\n    backbones = train_suite(base, \'backbones\')\n    winner = backbones.sort_values([\'macro_f1\', \'exp_id\'], ascending=[False, True]).iloc[0]\n    source = restore_config(winner.exp_id, base.seed)\n    selected = replace(base, backbone=source.backbone)\n    reuse_run(source, replace(selected, exp_id=\'T00\'))\n    print(\'Selected backbone on validation:\', selected.backbone, flush=True)\n    training = train_suite(selected, \'training\')\n    baseline_f1 = float(training.loc[training.exp_id == \'T00\', \'macro_f1\'].iloc[0])\n    # Choose at most one winning value per axis, without combining incompatible losses/mixes.\n    changes = {}\n    axes = [(\'T03_color\',), (\'T04_cutmix\', \'T05_mixup\'),\n            (\'T06_ls\', \'T07_focal\', \'T08_weighted\'), (\'T09_ema\',)]\n    for ids in axes:\n        best_axis = training[training.exp_id.isin(ids)].sort_values(\n            [\'macro_f1\', \'exp_id\'], ascending=[False, True]).iloc[0]\n        if best_axis.macro_f1 > baseline_f1:\n            changes.update(TRAINING_RECIPES[best_axis.exp_id])\n    combined = replace(selected, exp_id=\'T10_combined\', **changes)\n    if not (run_dir(combined) / \'summary.json\').exists():\n        if changes:\n            run(combined)\n        else:\n            reuse_run(replace(selected, exp_id=\'T00\'), combined)\n    combined_row = json.loads((run_dir(combined) / \'summary.json\').read_text())\n    training = pd.concat([training, pd.DataFrame([combined_row])], ignore_index=True)\n    training.to_csv(\'training_results.csv\', index=False)\n    best_row = training.sort_values([\'macro_f1\', \'exp_id\'], ascending=[False, True]).iloc[0]\n    best = restore_config(best_row.exp_id, base.seed)\n    inference = inference_suite(best, loaders(best, [val_df])[0])\n    inference[[\'exp_id\', \'gpu\', \'dtype\', \'batch\', \'p50\', \'p95\', \'p99\', \'images_per_s\']].to_csv(\n        \'latency_results.csv\', index=False)\n    import matplotlib.pyplot as plt\n    ax = inference.plot.scatter(x=\'p95\', y=\'macro_f1\', title=\'Validation F1 vs latency\')\n    for _, row in inference.iterrows():\n        ax.annotate(row.exp_id, (row.p95, row.macro_f1))\n    ax.figure.savefig(\'curves/inference_tradeoff.png\', dpi=150)\n    plt.close(ax.figure)\n    # Lock a supported final method before touching test. Prefer low ECE then latency on F1 ties.\n    eligible = inference[inference.exp_id.isin([\'I00\', \'I01_hflip_prob\', \'I02_hflip_logit\', \'I03_temperature\'])]\n    choice = eligible.sort_values([\'macro_f1\', \'ece\', \'p95\', \'exp_id\'],\n                                  ascending=[False, True, True, True]).iloc[0]\n    method = {\'I00\': \'identity\', \'I01_hflip_prob\': \'hflip_prob\',\n              \'I02_hflip_logit\': \'hflip_logit\', \'I03_temperature\': \'identity\'}[choice.exp_id]\n    lock = {\'recipe\': asdict(best), \'method\': method, \'calibrated\': True,\n            \'inference_choice\': choice.exp_id, \'seeds\': [0, 1, 2],\n            \'selection_split\': \'val\', \'test_used_for_selection\': False}\n    lock_path = Path(\'selection.json\')\n    if lock_path.exists() and json.loads(lock_path.read_text()) != lock:\n        raise ValueError(\'Selection is already locked; do not change after test\')\n    lock_path.write_text(json.dumps(lock, indent=2))\n    print(\'Locked selection:\', lock, flush=True)\n    rows = []\n    for seed in (0, 1, 2):\n        for original, exp_id, infer_method, calibrated in (\n                (best, \'F01\', method, True),\n                (replace(selected, exp_id=\'T00\'), \'T00_final\', \'identity\', False)):\n            cfg = replace(original, exp_id=exp_id, seed=seed, save_test_predictions=False)\n            if not (run_dir(cfg) / \'summary.json\').exists():\n                if seed == base.seed:\n                    reuse_run(original, cfg)\n                else:\n                    run(cfg)\n            result_path = run_dir(cfg) / \'final_metrics.json\'\n            prediction_path = Path(cfg.pred_dir) / f\'{exp_id}_seed{seed}_test.csv\'\n            if result_path.exists():\n                result = json.loads(result_path.read_text())\n            elif prediction_path.exists():\n                # Recover metrics from saved predictions without rerunning test inference.\n                df = pd.read_csv(prediction_path)\n                metrics = compute_metrics(df.y_true.to_numpy(), df.y_pred.to_numpy(),\n                                          df[[f\'p{i}\' for i in range(9)]].to_numpy())\n                result = {k: metrics[k] for k in (\'top1\', \'macro_f1\', \'ece\', \'balanced_acc\')}\n                result.update(exp_id=exp_id, seed=seed, method=infer_method)\n                result_path.write_text(json.dumps(result, indent=2))\n            else:\n                result = final_inference(cfg, *loaders(cfg, [val_df, test_df]),\n                                         method=infer_method, calibrated=calibrated)\n                result_path.write_text(json.dumps(result, indent=2))\n            rows.append(result)\n            pd.DataFrame(rows).to_csv(\'final_results.csv\', index=False)\n    for tag in (\'F01\', \'T00_final\'):\n        assert eval_main([\'score\', \'--pred\', f\'predictions/{tag}_seed*_test.csv\',\n                          \'--test-csv\', f\'{base.labels_dir}/test_subset0.csv\',\n                          \'--labels\', f\'{base.labels_dir}/labels.csv\', \'--tag\', tag,\n                          \'--out\', \'eval_out\']) == 0\n    assert eval_main([\'grade\', \'--final\', \'predictions/F01_seed*_test.csv\',\n                      \'--baseline\', \'predictions/T00_final_seed*_test.csv\',\n                      \'--uncal\', \'predictions/F01uncal_seed*_test.csv\',\n                      \'--final-val\', \'predictions/F01_seed*_val.csv\',\n                      \'--test-csv\', f\'{base.labels_dir}/test_subset0.csv\',\n                      \'--val-csv\', f\'{base.labels_dir}/val_subset0.csv\',\n                      \'--latency-p95-ms\', str(choice.p95), \'--out\', \'eval_out\']) == 0\n    pd.read_csv(\'eval_out/F01_per_class.csv\').to_csv(\'per_class_results.csv\', index=False)\n    export_results()\n    final = pd.DataFrame(rows)\n    stats = final.groupby(\'exp_id\')[[\'macro_f1\', \'top1\', \'ece\', \'balanced_acc\']].agg([\'mean\', \'std\'])\n    stats.to_csv(\'final_mean_std.csv\')\n    # Render confusion matrices from the actual saved predictions.\n    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay\n    for seed in (0, 1, 2):\n        df = pd.read_csv(f\'predictions/F01_seed{seed}_test.csv\')\n        fig, ax = plt.subplots(figsize=(8, 8))\n        ConfusionMatrixDisplay(confusion_matrix(df.y_true, df.y_pred, labels=list(range(9)))).plot(ax=ax)\n        fig.savefig(f\'curves/F01_seed{seed}_confusion.png\', dpi=150)\n        plt.close(fig)\n    report = (\'# DeepWeeds — kết quả chạy đầy đủ\\n\\n\'\n              \'Fold 0 chính thức; ảnh 224×224; batch 64; 12 epoch; AMP; chọn checkpoint bằng macro-F1 val. \'\n              \'Chọn backbone, recipe và suy luận chỉ trên val; test một lần cho mỗi cấu hình/seed.\\n\\n\'\n              f\'Backbone đã chọn: **{selected.backbone}**. Recipe thắng: **{best.exp_id}**. \'\n              f\'Suy luận: **{method}**, temperature khớp riêng trên val của từng seed.\\n\\n\'\n              \'## Backbone (validation)\\n\\n\' + backbones.to_csv(index=False) + \'\\n\'\n              \'## Ablation (validation)\\n\\n\' + training.to_csv(index=False) + \'\\n\'\n              \'## Suy luận (validation)\\n\\n\' + inference.to_csv(index=False) + \'\\n\'\n              \'## Chung kết — mean và std mẫu qua 3 seed\\n\\n\' + stats.to_string() + \'\\n\\n\'\n              \'## Hạn chế\\n\\n12 epoch có thể chưa đủ cho scratch. Dữ liệu mất cân bằng; \'\n              \'macro-F1 được ưu tiên hơn accuracy. Split ngẫu nhiên chưa đo tổng quát sang địa điểm mới. \'\n              \'Độ trễ chỉ gồm forward, chưa gồm đọc ảnh và gộp xác suất. \'\n              \'ConvNeXt dùng LayerNorm nên phép gộp Conv-BN có thể là no-op; FP16 được đo riêng. \'\n              \'GMAC từ thop có thể thiếu phép toán attention. Seed 0 được tái sử dụng khi recipe giống hệt, \'\n              \'nguồn ghi trong config.json; không phải một lần chạy độc lập bổ sung.\\n\')\n    Path(\'report.md\').write_text(report, encoding=\'utf-8\')\n    Path(\'COMPLETE.json\').write_text(json.dumps({\'status\': \'complete\', \'final_runs\': len(rows),\n                                              \'selection\': lock}, indent=2))\n    print(stats, flush=True)\n    print(\'ALL STAGES COMPLETE: results.xlsx, report.md, predictions/, curves/, eval_out/\', flush=True)\n    return final\n', 'inference.py': '"""Validation-only method selection; temperature learned on validation."""\nimport copy\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\ndef softmax(logits):\n    z = np.asarray(logits, dtype=np.float64)\n    z = z - z.max(axis=1, keepdims=True)\n    e = np.exp(z)\n    return e / e.sum(axis=1, keepdims=True)\n\ndef predict_logits(model, loader, device, view=None):\n    model.eval()\n    names, labels, logits = [], [], []\n    with torch.inference_mode():\n        for x, y, filenames in loader:\n            x = x.to(device=device, dtype=next(model.parameters()).dtype)\n            logits.append(model(x if view is None else view(x)).float().cpu().numpy())\n            labels.append(y.numpy())\n            names.extend(filenames)\n    if not labels:\n        raise ValueError(\'Empty evaluation loader\')\n    return names, np.concatenate(labels), np.concatenate(logits)\n\ndef view_identity(x):\n    return x\n\ndef view_hflip(x):\n    return x.flip(-1)\n\ndef views_multicrop(x, crop):\n    h, w = x.shape[-2:]\n    if not 0 < crop <= min(h, w):\n        raise ValueError(\'Crop must fit inside input\')\n    return [x[..., y:y+crop, z:z+crop] for y, z in\n            ((0, 0), (0, w-crop), (h-crop, 0), (h-crop, w-crop), ((h-crop)//2, (w-crop)//2))]\n\ndef views_multiscale(x, sizes):\n    return [F.interpolate(x, size=(s, s), mode=\'bilinear\', align_corners=False, antialias=True) for s in sizes]\n\ndef aggregate_views(logits_per_view, space=\'prob\'):\n    if not logits_per_view:\n        raise ValueError(\'At least one view is required\')\n    z = np.stack(logits_per_view)\n    if space == \'prob\':\n        return np.mean([softmax(v) for v in z], axis=0)\n    if space == \'logit\':\n        return softmax(z.mean(axis=0))\n    raise ValueError(\'space must be prob or logit\')\n\ndef ensemble_probs(list_of_probs):\n    p = np.stack(list_of_probs).mean(axis=0)\n    if not np.isfinite(p).all() or (p < 0).any() or not np.allclose(p.sum(1), 1, atol=1e-5):\n        raise ValueError(\'Ensemble requires normalized probabilities\')\n    return p\n\ndef fit_temperature(val_logits, val_labels):\n    z = torch.as_tensor(val_logits, dtype=torch.float64)\n    y = torch.as_tensor(val_labels, dtype=torch.long)\n    if z.ndim != 2 or len(z) != len(y) or not len(y) or not torch.isfinite(z).all():\n        raise ValueError(\'Invalid validation logits/labels\')\n    # Bounded scalar search is deterministic and cannot diverge like unconstrained LBFGS.\n    a, b = np.log(.05), np.log(20.)\n    ratio = (np.sqrt(5) - 1) / 2\n    def objective(logt):\n        return F.cross_entropy(z / np.exp(logt), y).item()\n    c, d = b-ratio*(b-a), a+ratio*(b-a)\n    fc, fd = objective(c), objective(d)\n    for _ in range(60):\n        if fc < fd:\n            b, d, fd = d, c, fc\n            c = b-ratio*(b-a)\n            fc = objective(c)\n        else:\n            a, c, fc = c, d, fd\n            d = a+ratio*(b-a)\n            fd = objective(d)\n    candidates = [0., a, b, (a+b)/2]\n    return float(np.exp(min(candidates, key=objective)))\n\ndef apply_temperature(logits, T):\n    if not np.isfinite(T) or T <= 0:\n        raise ValueError(\'Temperature must be finite and positive\')\n    return softmax(np.asarray(logits) / T)\n\ndef fuse_conv_bn(model):\n    fused = copy.deepcopy(model).eval()\n    def visit(module):\n        # Only fuse pairs with established forward adjacency: Sequential and timm ResNet blocks.\n        if isinstance(module, nn.Sequential):\n            children = list(module._modules.items())\n            for (a, conv), (b, bn) in zip(children, children[1:]):\n                if isinstance(conv, nn.Conv2d) and isinstance(bn, nn.BatchNorm2d):\n                    module._modules[a] = torch.nn.utils.fuse_conv_bn_eval(conv, bn)\n                    module._modules[b] = nn.Identity()\n        if module.__class__.__module__.startswith(\'timm.models.resnet\'):\n            for a, b in ((\'conv1\', \'bn1\'), (\'conv2\', \'bn2\'), (\'conv3\', \'bn3\')):\n                conv, bn = getattr(module, a, None), getattr(module, b, None)\n                if isinstance(conv, nn.Conv2d) and isinstance(bn, nn.BatchNorm2d):\n                    setattr(module, a, torch.nn.utils.fuse_conv_bn_eval(conv, bn))\n                    setattr(module, b, nn.Identity())\n        for child in module.children():\n            visit(child)\n    visit(fused)\n    return fused\n', 'losses.py': '"""Losses, class balancing, Mixup and area-corrected CutMix."""\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass LabelSmoothingCE(nn.CrossEntropyLoss):\n    def __init__(self, smoothing=.1):\n        super().__init__(label_smoothing=smoothing)\n\nclass FocalLoss(nn.Module):\n    def __init__(self, gamma=2., alpha=None):\n        super().__init__()\n        if gamma < 0:\n            raise ValueError(\'gamma must be nonnegative\')\n        self.gamma = gamma\n        self.register_buffer(\'alpha\', None if alpha is None else torch.as_tensor(alpha, dtype=torch.float32))\n    def forward(self, logits, target):\n        logpt = F.log_softmax(logits, dim=1).gather(1, target[:, None]).squeeze(1)\n        values = -(1 - logpt.exp()).pow(self.gamma) * logpt\n        if self.alpha is not None:\n            values = values * self.alpha[target]\n        return values.mean()\n\ndef build_criterion(kind=\'ce\', **kw):\n    if kind == \'ce\':\n        return nn.CrossEntropyLoss()\n    if kind == \'ls\':\n        return LabelSmoothingCE(kw.get(\'smoothing\', .1))\n    if kind == \'focal\':\n        return FocalLoss(kw.get(\'gamma\', 2.), kw.get(\'alpha\'))\n    if kind == \'ce_weighted\':\n        if kw.get(\'weight\') is None:\n            raise ValueError(\'ce_weighted requires train class weights\')\n        return nn.CrossEntropyLoss(weight=kw[\'weight\'])\n    raise ValueError(f\'Unknown loss: {kind}\')\n\ndef class_weights(counts, beta=0.):\n    n = torch.as_tensor(counts, dtype=torch.float64)\n    if (n <= 0).any() or not 0 <= beta < 1:\n        raise ValueError(\'Counts must be positive and beta in [0, 1)\')\n    w = 1 / n if beta == 0 else (1 - beta) / (-torch.expm1(n * np.log(beta)))\n    return (w / w.mean()).float()\n\ndef mix_batch(x, y, alpha=1., mode=\'cutmix\'):\n    if mode not in (\'mixup\', \'cutmix\'):\n        raise ValueError(f\'Unknown mixing mode: {mode}\')\n    if alpha <= 0:\n        return x, (y, y, 1.)\n    lam = float(np.random.beta(alpha, alpha))\n    perm = torch.randperm(len(x), device=x.device)\n    if mode == \'mixup\':\n        mixed = lam * x + (1 - lam) * x[perm]\n    else:\n        h, w = x.shape[-2:]\n        cut_h, cut_w = int(h * np.sqrt(1 - lam)), int(w * np.sqrt(1 - lam))\n        cy, cx = np.random.randint(h), np.random.randint(w)\n        y1, y2 = max(0, cy - cut_h // 2), min(h, cy + (cut_h + 1) // 2)\n        x1, x2 = max(0, cx - cut_w // 2), min(w, cx + (cut_w + 1) // 2)\n        mixed = x.clone()\n        mixed[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]\n        lam = 1 - (y2 - y1) * (x2 - x1) / (h * w)\n    return mixed, (y, y[perm], lam)\n\ndef mixed_loss(criterion, logits, targets):\n    a, b, lam = targets\n    return lam * criterion(logits, a) + (1 - lam) * criterion(logits, b)\n', 'model.py': '"""timm models, frozen backbones and optimizer groups."""\nimport copy\nimport torch\nfrom torch import nn\n\nSUGGESTED_BACKBONES = {\'resnet50\': \'resnet50\', \'resnext50\': \'resnext50_32x4d\', \'convnext_tiny\': \'convnext_tiny\',\n                       \'deit_small\': \'deit_small_patch16_224\', \'swin_tiny\': \'swin_tiny_patch4_window7_224\',\n                       \'efficientnet_b0\': \'efficientnet_b0\', \'mobilenetv3\': \'mobilenetv3_large_100\'}\n\ndef build_model(name, pretrained=True, num_classes=9, drop_rate=0., init=\'finetune\'):\n    import timm\n    if init not in (\'scratch\', \'frozen\', \'finetune\'):\n        raise ValueError(f\'Unknown initialization: {init}\')\n    model = timm.create_model(SUGGESTED_BACKBONES.get(name, name), pretrained=pretrained and init != \'scratch\',\n                              num_classes=num_classes, drop_rate=drop_rate)\n    if init == \'frozen\':\n        freeze_backbone(model)\n    return model\n\ndef freeze_backbone(model):\n    for p in model.parameters():\n        p.requires_grad_(False)\n    for p in model.get_classifier().parameters():\n        p.requires_grad_(True)\n    model.eval()\n    model.get_classifier().train()\n\ndef param_groups(model, lr_backbone, lr_head, weight_decay):\n    head = {id(p) for p in model.get_classifier().parameters()}\n    # Separate head norm/bias too: no weight decay on any norm/bias.\n    groups = {(is_head, decay): [] for is_head in (False, True) for decay in (False, True)}\n    no_decay = set(model.no_weight_decay()) if hasattr(model, \'no_weight_decay\') else set()\n    for name, p in model.named_parameters():\n        if p.requires_grad:\n            groups[(id(p) in head, p.ndim > 1 and not name.endswith(\'.bias\') and name not in no_decay)].append(p)\n    return [{\'params\': ps, \'lr\': lr_head if h else lr_backbone, \'weight_decay\': weight_decay if d else 0.}\n            for (h, d), ps in groups.items() if ps]\n\ndef count_params(model):\n    return sum(p.numel() for p in model.parameters()) / 1e6\n\ndef count_gmacs(model, img_size=224):\n    # thop may omit some attention ops: label the counting tool in the report.\n    from thop import profile\n    clone = copy.deepcopy(model).cpu().eval()\n    with torch.inference_mode():\n        macs, _ = profile(clone, inputs=(torch.zeros(1, 3, img_size, img_size),), verbose=False)\n    return macs / 1e9\n', 'test_core.py': '"""CPU tests for mathematical invariants; run python -m unittest test_core -v."""\nimport unittest\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom losses import FocalLoss, LabelSmoothingCE, mix_batch, class_weights\nfrom inference import fuse_conv_bn, fit_temperature, apply_temperature, aggregate_views\nfrom model import param_groups\nfrom train import Config, EMA, build_optimizer, build_scheduler, train_one_epoch, evaluate, parse_overrides\n\nclass CoreTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(4)\n        np.random.seed(4)\n    def test_focal_zero_and_smoothing_zero_equal_ce(self):\n        z, y = torch.randn(16, 9), torch.randint(9, (16,))\n        expected = nn.CrossEntropyLoss()(z, y)\n        torch.testing.assert_close(FocalLoss(0)(z, y), expected, atol=1e-6, rtol=0)\n        torch.testing.assert_close(LabelSmoothingCE(0)(z, y), expected)\n    def test_cutmix_lambda_matches_actual_area(self):\n        x = torch.stack([torch.full((3, 11, 13), float(i)) for i in range(8)])\n        y = torch.arange(8)\n        mixed, (a, b, lam) = mix_batch(x, y)\n        for i in range(8):\n            if a[i] != b[i]:\n                fraction = (mixed[i] != x[i]).float().mean().item()\n                self.assertAlmostEqual(fraction, 1-lam, places=6)\n        torch.testing.assert_close(x[:, 0, 0, 0], y.float())\n    def test_fusion_preserves_output_and_original(self):\n        model = nn.Sequential(nn.Conv2d(3, 5, 3), nn.BatchNorm2d(5), nn.ReLU()).eval()\n        model[1].running_mean.copy_(torch.randn(5))\n        model[1].running_var.copy_(torch.rand(5)+.2)\n        x = torch.randn(2, 3, 8, 8)\n        fused = fuse_conv_bn(model)\n        torch.testing.assert_close(model(x), fused(x), atol=1e-5, rtol=1e-5)\n        self.assertIsInstance(model[1], nn.BatchNorm2d)\n        self.assertIsInstance(fused[1], nn.Identity)\n    def test_temperature_does_not_change_argmax_and_reduces_nll(self):\n        rng = np.random.default_rng(2)\n        z, y = rng.normal(size=(100, 9))*5, rng.integers(9, size=100)\n        t = fit_temperature(z, y)\n        p, before = apply_temperature(z, t), apply_temperature(z, 1)\n        np.testing.assert_array_equal(p.argmax(1), z.argmax(1))\n        self.assertLessEqual(-np.log(p[np.arange(100), y]).mean(), -np.log(before[np.arange(100), y]).mean()+1e-8)\n    def test_weight_normalization_and_tta(self):\n        self.assertAlmostEqual(class_weights(range(1, 10), .999).mean().item(), 1., places=6)\n        z = np.random.randn(10, 9)\n        for space in (\'prob\', \'logit\'):\n            p = aggregate_views([z, z], space)\n            np.testing.assert_allclose(p.sum(1), 1)\n    def test_parameter_groups_cover_once_and_exclude_bias_decay(self):\n        class Model(nn.Module):\n            def __init__(self):\n                super().__init__()\n                self.backbone = nn.Linear(3, 4)\n                self.head = nn.Linear(4, 9)\n            def get_classifier(self):\n                return self.head\n        model = Model()\n        groups = param_groups(model, .001, .01, .05)\n        ids = [id(p) for g in groups for p in g[\'params\']]\n        self.assertEqual(len(ids), len(set(ids)))\n        self.assertEqual(set(ids), {id(p) for p in model.parameters()})\n        for g in groups:\n            for p in g[\'params\']:\n                if p.ndim == 1:\n                    self.assertEqual(g[\'weight_decay\'], 0)\n    def test_training_updates_head_and_keeps_frozen_bn(self):\n        class Toy(nn.Module):\n            def __init__(self):\n                super().__init__()\n                self.backbone = nn.Sequential(nn.Linear(3, 4), nn.BatchNorm1d(4))\n                self.head = nn.Linear(4, 9)\n            def get_classifier(self):\n                return self.head\n            def forward(self, x):\n                return self.head(self.backbone(x))\n        from model import freeze_backbone\n        model = Toy()\n        freeze_backbone(model)\n        before = model.head.weight.detach().clone()\n        bn_before = model.backbone[1].running_mean.clone()\n        loader = [(torch.randn(8, 3), torch.arange(8), [f\'{i}.jpg\' for i in range(8)])]\n        cfg = Config(init=\'frozen\', epochs=2, amp=False)\n        optimizer = build_optimizer(model, cfg)\n        scheduler = build_scheduler(optimizer, cfg, len(loader))\n        ema = EMA(model, .9)\n        scaler = torch.amp.GradScaler(\'cuda\', enabled=False)\n        row = train_one_epoch(model, loader, nn.CrossEntropyLoss(), optimizer, scheduler, scaler, cfg, torch.device(\'cpu\'), ema)\n        self.assertTrue(np.isfinite(row[\'train_loss\']))\n        self.assertFalse(torch.equal(before, model.head.weight))\n        torch.testing.assert_close(bn_before, model.backbone[1].running_mean)\n        ns, y, z, loss = evaluate(ema.model, loader, nn.CrossEntropyLoss(), torch.device(\'cpu\'))\n        self.assertEqual(z.shape, (8, 9))\n        self.assertEqual(ns, loader[0][2])\n        self.assertTrue(np.isfinite(loss))\n    def test_cli_types(self):\n        values = parse_overrides([\'seed=2\', \'amp=false\', \'ema_decay=0.99\', \'sampler=none\'])\n        self.assertEqual(values, {\'seed\': 2, \'amp\': False, \'ema_decay\': .99, \'sampler\': None})\n        with self.assertRaises(ValueError):\n            parse_overrides([\'amp=nope\'])\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'train.py': '"""One configurable training loop for B/T/F experiments; test is opt-in."""\nfrom dataclasses import dataclass, asdict, fields\nfrom pathlib import Path\nimport argparse\nimport copy\nimport json\nimport math\nimport platform\nimport random\nimport time\nimport sys\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom model import build_model, param_groups, count_params, count_gmacs\nfrom losses import build_criterion, class_weights, mix_batch, mixed_loss\nfrom inference import softmax\nif not (Path(__file__).parent / \'eval.py\').exists():\n    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom eval import compute_metrics, save_predictions\n\n@dataclass\nclass Config:\n    exp_id: str = \'T00\'\n    seed: int = 0\n    fold: int = 0\n    backbone: str = \'resnet50\'\n    init: str = \'finetune\'\n    drop_rate: float = 0.\n    img_size: int = 224\n    aug: str = \'basic\'\n    sampler: str | None = None\n    mix: str | None = None\n    mix_alpha: float = 1.\n    loss: str = \'ce\'\n    label_smoothing: float = .1\n    focal_gamma: float = 2.\n    class_weight_beta: float | None = None\n    epochs: int = 12\n    batch_size: int = 64\n    lr_backbone: float = 1e-4\n    lr_head: float = 1e-3\n    weight_decay: float = .05\n    warmup_epochs: float = 1.\n    ema_decay: float | None = None\n    amp: bool = True\n    num_workers: int = 2\n    images_dir: str = \'data/images\'\n    labels_dir: str = \'data/labels\'\n    out_dir: str = \'runs\'\n    pred_dir: str = \'predictions\'\n    curves_dir: str = \'curves\'\n    save_test_predictions: bool = False\n    pretrained: bool = True\n\ndef run_dir(cfg):\n    return Path(cfg.out_dir) / cfg.exp_id / f\'seed{cfg.seed}\'\n\ndef pred_path(cfg, split):\n    if split not in (\'val\', \'test\'):\n        raise ValueError(\'split must be val or test\')\n    return Path(cfg.pred_dir) / f\'{cfg.exp_id}_seed{cfg.seed}_{split}.csv\'\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\ndef build_optimizer(model, cfg):\n    return torch.optim.AdamW(param_groups(model, cfg.lr_backbone, cfg.lr_head, cfg.weight_decay))\n\ndef build_scheduler(optimizer, cfg, steps_per_epoch):\n    total = cfg.epochs * steps_per_epoch\n    warmup = min(round(cfg.warmup_epochs * steps_per_epoch), total)\n    def factor(step):\n        if step < warmup:\n            return (step + 1) / max(1, warmup)\n        progress = min(1., (step - warmup) / max(1, total - warmup))\n        return .5 * (1 + math.cos(math.pi * progress))\n    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)\n\nclass EMA:\n    def __init__(self, model, decay):\n        if not 0 <= decay < 1:\n            raise ValueError(\'EMA decay must be in [0,1)\')\n        self.model, self.decay = copy.deepcopy(model).eval(), decay\n        self.model.requires_grad_(False)\n    @torch.no_grad()\n    def update(self, model):\n        current = dict(model.named_parameters())\n        for name, p in self.model.named_parameters():\n            p.lerp_(current[name].detach(), 1-self.decay)\n        # Copy BN buffers from the live model; do not average integer counters.\n        buffers = dict(model.named_buffers())\n        for name, b in self.model.named_buffers():\n            b.copy_(buffers[name])\n\ndef train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, cfg, device, ema=None):\n    model.train()\n    if cfg.init == \'frozen\':\n        model.eval()\n        model.get_classifier().train()\n    total, n = 0., 0\n    for x, y, _ in loader:\n        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n        targets = y\n        if cfg.mix:\n            x, targets = mix_batch(x, y, cfg.mix_alpha, cfg.mix)\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast(device.type, enabled=cfg.amp and device.type == \'cuda\'):\n            logits = model(x)\n            loss = mixed_loss(criterion, logits, targets) if cfg.mix else criterion(logits, targets)\n        if not torch.isfinite(loss):\n            raise RuntimeError(\'Nonfinite training loss\')\n        scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)\n        old_scale = scaler.get_scale()\n        scaler.step(optimizer)\n        scaler.update()\n        # A scaler decrease means the optimizer step was skipped for overflow.\n        if scaler.get_scale() >= old_scale:\n            scheduler.step()\n            if ema:\n                ema.update(model)\n        total += loss.item() * len(y)\n        n += len(y)\n    if n == 0:\n        raise ValueError(\'Empty training loader\')\n    return {\'train_loss\': total/n, \'lr\': optimizer.param_groups[0][\'lr\']}\n\ndef evaluate(model, loader, criterion, device):\n    model.eval()\n    names, ys, zs, total = [], [], [], 0.\n    with torch.inference_mode():\n        for x, y, filenames in loader:\n            x, y = x.to(device), y.to(device)\n            z = model(x)\n            total += criterion(z, y).item() * len(y)\n            names.extend(filenames)\n            ys.append(y.cpu().numpy())\n            zs.append(z.float().cpu().numpy())\n    if not ys:\n        raise ValueError(\'Empty evaluation loader\')\n    return names, np.concatenate(ys), np.concatenate(zs), total/len(names)\n\ndef plot_curves(history, path, title):\n    import matplotlib.pyplot as plt\n    df = pd.DataFrame(history)\n    fig, axs = plt.subplots(1, 3, figsize=(14, 4))\n    axs[0].plot(df.epoch, df.train_loss, label=\'train\')\n    axs[0].plot(df.epoch, df.val_loss, label=\'val\')\n    axs[0].set_ylabel(\'Loss\')\n    axs[0].legend()\n    axs[1].plot(df.epoch, df.macro_f1, label=\'val macro-F1\')\n    axs[1].plot(df.epoch, df.top1, label=\'val top-1\')\n    axs[1].legend()\n    axs[2].plot(df.epoch, df.lr)\n    axs[2].set_ylabel(\'Backbone LR\')\n    for ax in axs:\n        ax.set_xlabel(\'Epoch\')\n        ax.grid(alpha=.2)\n    fig.suptitle(title)\n    fig.tight_layout()\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, dpi=150)\n    plt.close(fig)\n\ndef run(cfg):\n    from dataset import load_split, check_split, make_loader, build_transforms\n    import timm\n    if cfg.epochs < 1 or cfg.batch_size < 1:\n        raise ValueError(\'epochs and batch_size must be positive\')\n    target = run_dir(cfg)\n    if (target / \'history.csv\').exists():\n        raise FileExistsError(f\'Run already exists: {target}; choose another exp_id\')\n    if cfg.save_test_predictions and pred_path(cfg, \'test\').exists():\n        raise FileExistsError(\'Test predictions already exist; do not repeat final evaluation\')\n    set_seed(cfg.seed)\n    target.mkdir(parents=True, exist_ok=True)\n    train, val, test = load_split(cfg.labels_dir, cfg.fold)\n    checks = check_split(train, val, test, cfg.images_dir)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = build_model(cfg.backbone, cfg.pretrained, drop_rate=cfg.drop_rate, init=cfg.init)\n    data_cfg = timm.data.resolve_model_data_config(model)\n    metadata = {**asdict(cfg), \'python\': platform.python_version(), \'torch\': str(torch.__version__),\n                \'timm\': timm.__version__, \'pretrained_cfg\': model.pretrained_cfg, \'data_cfg\': data_cfg,\n                \'split_checks\': checks, \'device\': str(device), \'gmac_tool\': \'thop (attention ops may be incomplete)\'}\n    (target / \'config.json\').write_text(json.dumps(metadata, indent=2, default=str), encoding=\'utf-8\')\n    transforms = {s: build_transforms(s == \'train\', cfg.img_size, cfg.aug, data_cfg[\'mean\'], data_cfg[\'std\'])\n                  for s in (\'train\', \'val\')}\n    def loader(df, training=False):\n        return make_loader(df, cfg.images_dir, transforms[\'train\' if training else \'val\'], cfg.batch_size,\n                           training, cfg.sampler if training else None, cfg.num_workers, cfg.seed)\n    train_loader, val_loader = loader(train, True), loader(val)\n    params = count_params(model)\n    try:\n        gmacs = count_gmacs(model, cfg.img_size)\n    except (ImportError, RuntimeError, AssertionError) as exc:\n        gmacs = None\n        print(\'GMAC unavailable:\', exc)\n    model = model.to(device)\n    weights = None\n    if cfg.loss == \'ce_weighted\' or cfg.class_weight_beta is not None:\n        weights = class_weights(train.Label.value_counts().reindex(range(9), fill_value=0).values,\n                                cfg.class_weight_beta or 0.).to(device)\n    criterion = build_criterion(cfg.loss, smoothing=cfg.label_smoothing, gamma=cfg.focal_gamma,\n                                alpha=weights, weight=weights).to(device)\n    optimizer = build_optimizer(model, cfg)\n    scheduler = build_scheduler(optimizer, cfg, len(train_loader))\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=cfg.amp and device.type == \'cuda\')\n    ema = EMA(model, cfg.ema_decay) if cfg.ema_decay is not None else None\n    history, best, best_epoch = [], -1., 0\n    start = time.perf_counter()\n    for epoch in range(1, cfg.epochs+1):\n        epoch_start = time.perf_counter()\n        row = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, cfg, device, ema)\n        selected = ema.model if ema else model\n        names, y, z, val_loss = evaluate(selected, val_loader, criterion, device)\n        p = softmax(z)\n        metric = compute_metrics(y, p.argmax(1), p)\n        row.update(epoch=epoch, val_loss=val_loss, macro_f1=metric[\'macro_f1\'], top1=metric[\'top1\'],\n                   ece=metric[\'ece\'], seconds=time.perf_counter()-epoch_start)\n        history.append(row)\n        pd.DataFrame(history).to_csv(target / \'history.csv\', index=False)\n        if metric[\'macro_f1\'] > best:\n            best, best_epoch = metric[\'macro_f1\'], epoch\n            torch.save({\'model\': selected.state_dict(), \'epoch\': epoch, \'config\': asdict(cfg)}, target / \'best.pt\')\n        torch.save({\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(),\n                    \'scaler\': scaler.state_dict(), \'ema\': ema.model.state_dict() if ema else None,\n                    \'epoch\': epoch, \'config\': asdict(cfg)}, target / \'last.pt\')\n        plot_curves(history, Path(cfg.curves_dir) / f\'{cfg.exp_id}_seed{cfg.seed}.png\', f\'{cfg.exp_id} / {cfg.backbone}\')\n        print(f"Epoch {epoch}/{cfg.epochs}: loss={row[\'train_loss\']:.4f} val F1={best:.4f} (best)", flush=True)\n    model.load_state_dict(torch.load(target / \'best.pt\', map_location=device, weights_only=True)[\'model\'])\n    for split, df in [(\'val\', val)] + ([(\'test\', test)] if cfg.save_test_predictions else []):\n        names, y, z, _ = evaluate(model, loader(df), criterion, device)\n        np.savez_compressed(target / f\'{split}_logits.npz\', filenames=np.array(names), y_true=y, logits=z)\n        save_predictions(pred_path(cfg, split), names, y, softmax(z))\n    result = {\'exp_id\': cfg.exp_id, \'backbone\': cfg.backbone, \'seed\': cfg.seed, \'best_epoch\': best_epoch,\n              \'macro_f1\': best, \'top1\': history[best_epoch-1][\'top1\'], \'params_m\': params, \'gmacs\': gmacs,\n              \'train_seconds\': time.perf_counter()-start, \'seconds_per_epoch\': np.mean([r[\'seconds\'] for r in history]),\n              \'checkpoint\': str(target / \'best.pt\')}\n    (target / \'summary.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    return result\n\ndef parse_overrides(pairs):\n    defaults = asdict(Config())\n    optional_numeric = {\'ema_decay\', \'class_weight_beta\'}\n    result = {}\n    for pair in pairs:\n        key, sep, value = pair.partition(\'=\')\n        if not sep or key not in defaults:\n            raise ValueError(f\'Invalid Config override: {pair}\')\n        default = defaults[key]\n        if value.lower() in (\'none\', \'null\'):\n            if default is not None:\n                raise ValueError(f\'{key} does not accept None\')\n            result[key] = None\n        elif isinstance(default, bool):\n            if value.lower() not in (\'true\', \'false\'):\n                raise ValueError(f\'{key} requires true/false\')\n            result[key] = value.lower() == \'true\'\n        elif isinstance(default, int):\n            result[key] = int(value)\n        elif isinstance(default, float) or key in optional_numeric:\n            result[key] = float(value)\n        else:\n            result[key] = value\n    return result\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--set\', nargs=\'*\', default=[])\n    args = parser.parse_args()\n    print(json.dumps(run(Config(**parse_overrides(args.set))), indent=2))\n\nif __name__ == \'__main__\':\n    main()\n', 'eval.py': '#!/usr/bin/env python3\n"""eval.py - công cụ đánh giá chính thức của Lab Day 2 (DeepWeeds).\n\nFile này đã hoàn chỉnh: sinh viên KHÔNG sửa công thức chỉ số. Mọi con số trong\nresults.xlsx và báo cáo phải khớp với kết quả của file này (RUBRIC.md, mục I).\nChỉ phụ thuộc numpy và pandas.\n\nĐịnh nghĩa chỉ số (README.md, mục 2.2)\n--------------------------------------\n- top-1 accuracy : số ảnh đoán đúng / tổng số ảnh, không trọng số theo lớp.\n- macro-F1       : trung bình cộng F1 của 9 lớp (lớp không có dự đoán đúng có F1 = 0).\n- balanced acc.  : trung bình recall của các lớp có ảnh trong tập.\n- precision / recall / F1 theo lớp, ma trận nhầm lẫn (hàng = nhãn thật).\n- ECE            : 15 bin đều theo độ tin cậy (= max softmax); bin m là (b_{m-1}, b_m].\n                   ECE = sum_m (n_m / n) * |acc_m - conf_m|\n- mean +- std    : qua các seed, std mẫu (ddof = 1).\n\nĐịnh dạng file dự đoán (mỗi seed một file)\n------------------------------------------\n    Filename, y_true, y_pred, p0, p1, ..., p8\np0..p8 là xác suất softmax, thứ tự theo cột `Label` của labels.csv\n(0 = Chinee Apple ... 7 = Snake Weed, 8 = Negatives). Tên file chứa `seed<k>`,\nví dụ predictions/F01_seed0_test.csv. Hàm `save_predictions(...)` trong file này ghi\nđúng định dạng đó (`from eval import save_predictions`).\n\nCách dùng\n---------\n    # 1) Tính chỉ số cho một nhóm file (một cấu hình, nhiều seed)\n    python eval.py score --pred "predictions/F01_seed*_test.csv" \\\n        --test-csv labels/test_subset0.csv --labels labels/labels.csv --tag F01\n\n    # 2) Tự chấm phần I của RUBRIC (chung kết so với mốc)\n    python eval.py grade \\\n        --final    "predictions/F01_seed*_test.csv" \\\n        --baseline "predictions/T00_seed*_test.csv" \\\n        --uncal    "predictions/F01_uncal_seed*_test.csv" \\\n        --final-val "predictions/F01_seed*_val.csv" \\\n        --latency-p95-ms 41.5 --latency-method proper \\\n        --test-csv labels/test_subset0.csv --labels labels/labels.csv\n\n`--uncal` (cùng cấu hình nhưng chưa temperature scaling) và `--final-val` (dự đoán\ntrên val) là tùy chọn; thiếu thì các ý I4(a) hoặc I4(b) ghi là "chưa chấm được".\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport glob\nimport json\nimport math\nimport re\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n# --------------------------------------------------------------------------- #\n# Hằng số\n# --------------------------------------------------------------------------- #\nNUM_CLASSES = 9\nECE_BINS = 15\nPROB_SUM_TOL = 1e-3  # xác suất mỗi dòng phải cộng bằng 1 (sai số cho phép)\n\n# Thứ tự lớp theo cột `Label` của labels.csv (lấy từ deepweeds.py của tác giả).\nCLASS_NAMES = [\n    "Chinee Apple", "Lantana", "Parkinsonia", "Parthenium", "Prickly Acacia",\n    "Rubber Vine", "Siam Weed", "Snake Weed", "Negatives",\n]\n\n# Hai lớp khó nhất và mốc recall (%) từ bài báo gốc (README.md, mục 2.3).\nHARD_CLASSES = {"Chinee Apple": 88.5, "Snake Weed": 88.8}\n\n# ---- Ngưỡng chấm RUBRIC mục I (TẠM THỜI: giảng viên có thể chỉnh tại đây) ---- #\nACC_TIERS = [(95.7, 7), (95.1, 6), (94.0, 5), (92.0, 3), (90.0, 1)]  # I1: (acc %, điểm)\nI2_MIN_DELTA = 0.01          # I2: Delta macro-F1 tối thiểu để đạt 5 điểm\nHARD_FLOORS = [(85.0, 3), (80.0, 2)]  # I3: sàn recall (%) cho cả hai lớp khó\nI3_MIN_POINTS = 1            # I3: có báo cáo đầy đủ nhưng dưới 80%\nI4_MAX_VAL_TEST_GAP = 0.02   # I4(b): |macro-F1 val - macro-F1 test| tối đa\nI5_P95_BUDGET_MS = 100.0     # I5: ngân sách p95 ở batch 1\nMIN_SEEDS = 3\n\n\n# --------------------------------------------------------------------------- #\n# Đọc và kiểm tra file dự đoán\n# --------------------------------------------------------------------------- #\n@dataclass\nclass Pred:\n    path: str\n    seed: int | None\n    filenames: np.ndarray\n    y_true: np.ndarray\n    y_pred: np.ndarray\n    probs: np.ndarray\n\n\ndef parse_seed(path: str) -> int | None:\n    m = re.search(r"seed[_-]?(\\d+)", Path(path).name)\n    return int(m.group(1)) if m else None\n\n\ndef read_pred(path: str, num_classes: int = NUM_CLASSES) -> Pred:\n    """Đọc một file dự đoán và kiểm tra định dạng. Ném ValueError nếu sai."""\n    df = pd.read_csv(path)\n    prob_cols = [f"p{i}" for i in range(num_classes)]\n    missing = [c for c in ["Filename", "y_true", "y_pred", *prob_cols] if c not in df.columns]\n    if missing:\n        raise ValueError(f"{path}: thiếu cột {missing}")\n    extra_p = [c for c in df.columns if re.fullmatch(r"p\\d+", c) and c not in prob_cols]\n    if extra_p:\n        raise ValueError(f"{path}: có cột xác suất ngoài p0..p{num_classes - 1}: {extra_p}")\n    if len(df) == 0:\n        raise ValueError(f"{path}: file rỗng")\n    if df["Filename"].duplicated().any():\n        raise ValueError(f"{path}: có Filename bị trùng")\n\n    probs = df[prob_cols].to_numpy(dtype=np.float64)\n    if not np.isfinite(probs).all() or (probs < 0).any() or (probs > 1 + 1e-6).any():\n        raise ValueError(f"{path}: xác suất chứa NaN, âm hoặc lớn hơn 1; hãy lưu softmax, không lưu logit")\n    bad_sum = np.abs(probs.sum(1) - 1.0) > PROB_SUM_TOL\n    if bad_sum.any():\n        raise ValueError(f"{path}: {int(bad_sum.sum())} dòng có tổng xác suất khác 1 "\n                         f"(sai số > {PROB_SUM_TOL}); hãy lưu softmax, không lưu logit")\n\n    ints = {}\n    for col in ("y_true", "y_pred"):\n        v = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)\n        if not np.isfinite(v).all() or (v != np.round(v)).any():\n            raise ValueError(f"{path}: cột {col} phải là số nguyên")\n        v = v.astype(np.int64)\n        if v.min() < 0 or v.max() >= num_classes:\n            raise ValueError(f"{path}: cột {col} ngoài khoảng 0..{num_classes - 1}")\n        ints[col] = v\n\n    argmax = probs.argmax(1)\n    mismatch = int((argmax != ints["y_pred"]).sum())\n    if mismatch:\n        raise ValueError(f"{path}: {mismatch} dòng có y_pred khác argmax của p0..p{num_classes - 1}")\n\n    return Pred(path, parse_seed(path), df["Filename"].to_numpy(), ints["y_true"], ints["y_pred"], probs)\n\n\ndef save_predictions(path: str | Path, filenames, y_true, probs, num_classes: int = NUM_CLASSES) -> Path:\n    """Ghi một file dự đoán đúng định dạng của eval.py: Filename, y_true, y_pred, p0..p{K-1}.\n\n    Dùng hàm này (hoặc ghi y hệt định dạng) để tạo predictions/<exp_id>_seed<k>_<split>.csv.\n    `probs` là XÁC SUẤT softmax (mỗi dòng cộng bằng 1), không phải logit.\n    y_pred được tính bằng argmax của probs. Kết quả luôn qua được `read_pred`.\n    """\n    probs = np.asarray(probs, dtype=np.float64)\n    y_true = np.asarray(y_true, dtype=np.int64)\n    filenames = list(filenames)\n    if probs.ndim != 2 or probs.shape[1] != num_classes:\n        raise ValueError(f"probs phải có dạng (N, {num_classes}), nhận {probs.shape}")\n    if not (len(filenames) == len(y_true) == len(probs)):\n        raise ValueError("filenames, y_true, probs phải cùng độ dài")\n    if np.abs(probs.sum(1) - 1.0).max() > PROB_SUM_TOL:\n        raise ValueError("probs chưa chuẩn hoá (tổng mỗi dòng khác 1); hãy áp dụng softmax trước khi lưu")\n    df = pd.DataFrame({"Filename": filenames, "y_true": y_true, "y_pred": probs.argmax(1)})\n    for i in range(num_classes):\n        df[f"p{i}"] = probs[:, i]\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(path, index=False, float_format="%.8g")\n    return path\n\n\ndef check_against_csv(pred: Pred, csv_path: str, what: str = "test") -> None:\n    """Đối chiếu tên file và nhãn thật với file CSV chia sẵn (Filename, Label)."""\n    ref = pd.read_csv(csv_path)\n    if not {"Filename", "Label"} <= set(ref.columns):\n        raise ValueError(f"{csv_path}: cần cột Filename và Label")\n    ref_map = dict(zip(ref["Filename"], ref["Label"].astype(int)))\n    names = set(pred.filenames)\n    if names != set(ref_map):\n        only_pred = len(names - set(ref_map))\n        only_ref = len(set(ref_map) - names)\n        raise ValueError(f"{pred.path}: không khớp {what} CSV ({csv_path}): "\n                         f"thừa {only_pred} ảnh, thiếu {only_ref} ảnh")\n    wrong = sum(int(ref_map[f]) != int(t) for f, t in zip(pred.filenames, pred.y_true))\n    if wrong:\n        raise ValueError(f"{pred.path}: {wrong} dòng có y_true khác Label trong {csv_path}")\n\n\ndef load_names(labels_csv: str | None) -> list[str]:\n    """Lấy tên lớp từ labels.csv (cột Label, Species) nếu có; nếu không dùng mặc định."""\n    if not labels_csv:\n        return list(CLASS_NAMES)\n    df = pd.read_csv(labels_csv)\n    if not {"Label", "Species"} <= set(df.columns):\n        raise ValueError(f"{labels_csv}: cần cột Label và Species")\n    m = df.drop_duplicates("Label").sort_values("Label")\n    if list(m["Label"]) != list(range(NUM_CLASSES)):\n        raise ValueError(f"{labels_csv}: Label phải là 0..{NUM_CLASSES - 1}")\n    return [str(s) for s in m["Species"]]\n\n\ndef _norm(name: str) -> str:\n    return re.sub(r"[^a-z]", "", name.lower())\n\n\ndef hard_class_indices(names: list[str]) -> dict[str, int]:\n    out = {}\n    for hard in HARD_CLASSES:\n        for i, n in enumerate(names):\n            if _norm(n) == _norm(hard):\n                out[hard] = i\n    missing = set(HARD_CLASSES) - set(out)\n    if missing:\n        raise ValueError(f"không tìm thấy lớp {sorted(missing)} trong danh sách tên lớp {names}")\n    return out\n\n\ndef expand(pattern: str | list[str]) -> list[str]:\n    patterns = [pattern] if isinstance(pattern, str) else list(pattern)\n    files: list[str] = []\n    for p in patterns:\n        files.extend(glob.glob(p))\n    files = sorted(set(files))\n    if not files:\n        raise FileNotFoundError(f"không có file nào khớp: {patterns}")\n    return files\n\n\n# --------------------------------------------------------------------------- #\n# Chỉ số\n# --------------------------------------------------------------------------- #\ndef confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, k: int = NUM_CLASSES) -> np.ndarray:\n    cm = np.zeros((k, k), dtype=np.int64)\n    np.add.at(cm, (y_true, y_pred), 1)\n    return cm\n\n\ndef per_class(cm: np.ndarray) -> dict[str, np.ndarray]:\n    tp = np.diag(cm).astype(np.float64)\n    support = cm.sum(1).astype(np.float64)\n    predicted = cm.sum(0).astype(np.float64)\n    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)\n    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)\n    denom = precision + recall\n    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)\n    return {"precision": precision, "recall": recall, "f1": f1, "support": support}\n\n\ndef ece_score(probs: np.ndarray, y_true: np.ndarray, bins: int = ECE_BINS) -> float:\n    """ECE với `bins` bin đều; độ tin cậy = max softmax; bin m là (b_{m-1}, b_m]."""\n    conf = probs.max(1)\n    correct = (probs.argmax(1) == y_true).astype(np.float64)\n    idx = np.clip(np.ceil(conf * bins).astype(np.int64) - 1, 0, bins - 1)\n    n = len(conf)\n    ece = 0.0\n    for m in range(bins):\n        mask = idx == m\n        if mask.any():\n            ece += mask.sum() / n * abs(correct[mask].mean() - conf[mask].mean())\n    return float(ece)\n\n\ndef compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, probs: np.ndarray,\n                    k: int = NUM_CLASSES, bins: int = ECE_BINS) -> dict:\n    cm = confusion_matrix(y_true, y_pred, k)\n    pc = per_class(cm)\n    has_support = pc["support"] > 0\n    nll = float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, None)).mean())\n    return {\n        "n": int(len(y_true)),\n        "top1": float((y_true == y_pred).mean()),\n        "macro_f1": float(pc["f1"].mean()),\n        "balanced_acc": float(pc["recall"][has_support].mean()),\n        "ece": ece_score(probs, y_true, bins),\n        "nll": nll,\n        "precision": pc["precision"], "recall": pc["recall"], "f1": pc["f1"],\n        "support": pc["support"], "confusion": cm,\n    }\n\n\nSCALARS = ("top1", "macro_f1", "balanced_acc", "ece", "nll")\nVECTORS = ("precision", "recall", "f1")\n\n\ndef mean_std(values) -> tuple[float, float]:\n    """mean và std mẫu (ddof=1) theo trục seed; std là NaN nếu chỉ có 1 seed."""\n    a = np.asarray(values, dtype=np.float64)\n    mean = a.mean(axis=0)\n    std = a.std(axis=0, ddof=1) if len(a) > 1 else np.full(a.shape[1:], np.nan)\n    if a.ndim == 1:\n        return float(mean), float(std)\n    return mean, std\n\n\n@dataclass\nclass Group:\n    preds: list[Pred]\n    metrics: list[dict]\n    summary: dict  # {"top1": (mean, std), ..., "recall": (vec_mean, vec_std), ...}\n\n    @property\n    def seeds(self) -> list[int | None]:\n        return [p.seed for p in self.preds]\n\n\ndef load_group(pattern, ref_csv: str | None, k: int = NUM_CLASSES, ref_what: str = "test") -> Group:\n    files = expand(pattern)\n    preds = [read_pred(f, k) for f in files]\n    if ref_csv:\n        for p in preds:\n            check_against_csv(p, ref_csv, ref_what)\n    base_names = set(preds[0].filenames)\n    for p in preds[1:]:\n        if set(p.filenames) != base_names:\n            raise ValueError(f"{p.path}: tập Filename khác {preds[0].path}")\n    seeds = [p.seed for p in preds if p.seed is not None]\n    if len(seeds) != len(set(seeds)):\n        raise ValueError(f"seed bị trùng giữa các file: {files}")\n    metrics = [compute_metrics(p.y_true, p.y_pred, p.probs, k) for p in preds]\n    summary = {key: mean_std([m[key] for m in metrics]) for key in (*SCALARS, *VECTORS)}\n    return Group(preds, metrics, summary)\n\n\n# --------------------------------------------------------------------------- #\n# Định dạng đầu ra\n# --------------------------------------------------------------------------- #\ndef fmt(mean: float, std: float, digits: int = 4) -> str:\n    if std is None or (isinstance(std, float) and math.isnan(std)):\n        return f"{mean:.{digits}f} (1 seed)"\n    return f"{mean:.{digits}f} ± {std:.{digits}f}"\n\n\ndef report_group(name: str, g: Group, names: list[str]) -> str:\n    lines = [f"### {name} ({len(g.preds)} seed: {g.seeds}; {g.metrics[0][\'n\']} ảnh)", ""]\n    if len(g.preds) < MIN_SEEDS:\n        lines += [f"> CẢNH BÁO: chỉ có {len(g.preds)} seed, yêu cầu tối thiểu {MIN_SEEDS}.", ""]\n    lines += ["| Chỉ số | mean ± std |", "|---|---|"]\n    labels = {"top1": "top-1 accuracy", "macro_f1": "macro-F1", "balanced_acc": "balanced accuracy",\n              "ece": "ECE (15 bin)", "nll": "NLL"}\n    for key in SCALARS:\n        lines.append(f"| {labels[key]} | {fmt(*g.summary[key])} |")\n    lines += ["", "| Lớp | Precision | Recall | F1 | Số ảnh |", "|---|---|---|---|---|"]\n    pm, ps = g.summary["precision"]\n    rm, rs = g.summary["recall"]\n    fm, fs = g.summary["f1"]\n    for i, n in enumerate(names):\n        lines.append(f"| {n} | {fmt(pm[i], ps[i], 3)} | {fmt(rm[i], rs[i], 3)} | "\n                     f"{fmt(fm[i], fs[i], 3)} | {int(g.metrics[0][\'support\'][i])} |")\n    return "\\n".join(lines)\n\n\ndef save_group(out_dir: Path, tag: str, g: Group, names: list[str]) -> None:\n    out_dir.mkdir(parents=True, exist_ok=True)\n    rows = []\n    for p, m in zip(g.preds, g.metrics):\n        rows.append({"file": Path(p.path).name, "seed": p.seed,\n                     **{k: m[k] for k in ("n", *SCALARS)}})\n    pd.DataFrame(rows).to_csv(out_dir / f"{tag}_per_seed.csv", index=False)\n    pc = pd.DataFrame({"class": names})\n    for key in VECTORS:\n        mean, std = g.summary[key]\n        pc[f"{key}_mean"], pc[f"{key}_std"] = mean, std\n    pc["support"] = g.metrics[0]["support"].astype(int)\n    pc.to_csv(out_dir / f"{tag}_per_class.csv", index=False)\n    cm = sum(m["confusion"] for m in g.metrics)\n    pd.DataFrame(cm, index=[f"true_{n}" for n in names],\n                 columns=[f"pred_{n}" for n in names]).to_csv(out_dir / f"{tag}_confusion_sum.csv")\n    summary = {"seeds": g.seeds,\n               **{k: {"mean": g.summary[k][0], "std": g.summary[k][1]} for k in SCALARS},\n               **{k: {"mean": g.summary[k][0].tolist(), "std": g.summary[k][1].tolist()} for k in VECTORS},\n               "classes": names}\n    (out_dir / f"{tag}_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))\n\n\n# --------------------------------------------------------------------------- #\n# Lệnh `score`\n# --------------------------------------------------------------------------- #\ndef cmd_score(args) -> int:\n    names = load_names(args.labels)\n    g = load_group(args.pred, args.test_csv, ref_what="test")\n    if not args.test_csv:\n        print("LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\\n")\n    print(report_group(args.tag, g, names))\n    if args.out:\n        save_group(Path(args.out), args.tag, g, names)\n        print(f"\\nĐã lưu kết quả vào {args.out}/{args.tag}_*")\n    return 0\n\n\n# --------------------------------------------------------------------------- #\n# Lệnh `grade` (RUBRIC mục I)\n# --------------------------------------------------------------------------- #\ndef points_i1(acc_pct: float) -> int:\n    for thr, pts in ACC_TIERS:\n        if acc_pct >= thr:\n            return pts\n    return 0\n\n\ndef points_i2(delta: float, s: float) -> int:\n    if not math.isfinite(delta) or delta <= 0:\n        return 0\n    s = 0.0 if not math.isfinite(s) else s\n    if delta > s and delta >= I2_MIN_DELTA:\n        return 5\n    return 4 if delta > s else 2\n\n\ndef points_i3(recalls_pct: dict[str, float]) -> int:\n    if all(recalls_pct[c] >= ref for c, ref in HARD_CLASSES.items()):\n        return 4\n    low = min(recalls_pct.values())\n    for floor, pts in HARD_FLOORS:\n        if low >= floor:\n            return pts\n    return I3_MIN_POINTS\n\n\ndef cmd_grade(args) -> int:\n    names = load_names(args.labels)\n    hard_idx = hard_class_indices(names)\n    final = load_group(args.final, args.test_csv, ref_what="test")\n    base = load_group(args.baseline, args.test_csv, ref_what="test")\n    if not args.test_csv:\n        print("LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\\n")\n    warnings: list[str] = []\n    for label, g in (("chung kết", final), ("mốc", base)):\n        if len(g.preds) < MIN_SEEDS:\n            warnings.append(f"nhóm {label} chỉ có {len(g.preds)} seed (< {MIN_SEEDS})")\n    if set(final.preds[0].filenames) != set(base.preds[0].filenames):\n        warnings.append("tập Filename của chung kết và mốc khác nhau")\n\n    items: list[tuple[str, str, int | None, int, str]] = []  # (mã, tiêu chí, điểm, tối đa, ghi chú)\n\n    # I1\n    acc = final.summary["top1"][0] * 100\n    items.append(("I1", "Top-1 accuracy test", points_i1(acc), 7, f"{acc:.2f}% (mean {len(final.preds)} seed)"))\n\n    # I2\n    mf_f, sd_f = final.summary["macro_f1"]\n    mf_b, sd_b = base.summary["macro_f1"]\n    delta = mf_f - mf_b\n    s = max([v for v in (sd_f, sd_b) if math.isfinite(v)], default=math.nan)\n    items.append(("I2", "Macro-F1 cải thiện so với mốc", points_i2(delta, s), 5,\n                  f"final {mf_f:.4f}, mốc {mf_b:.4f}, Δ={delta:+.4f}, s={s:.4f}"))\n\n    # I3\n    rec_mean = final.summary["recall"][0] * 100\n    recalls = {c: float(rec_mean[i]) for c, i in hard_idx.items()}\n    note = ", ".join(f"{c} {recalls[c]:.1f}% (mốc {HARD_CLASSES[c]}%)" for c in HARD_CLASSES)\n    items.append(("I3", "Recall hai lớp khó", points_i3(recalls), 4, note))\n\n    # I4(a)\n    if args.uncal:\n        uncal = load_group(args.uncal, args.test_csv, ref_what="test")\n        e_after, e_before = final.summary["ece"][0], uncal.summary["ece"][0]\n        items.append(("I4a", "ECE sau TS < ECE trước", int(e_after < e_before), 1,\n                      f"trước {e_before:.4f}, sau {e_after:.4f}"))\n    else:\n        items.append(("I4a", "ECE sau TS < ECE trước", None, 1, "chưa chấm được (thiếu --uncal)"))\n\n    # I4(b)\n    if args.final_val:\n        fv = load_group(args.final_val, args.val_csv, ref_what="val")\n        gap = abs(fv.summary["macro_f1"][0] - mf_f)\n        items.append(("I4b", "Chênh macro-F1 val/test <= %.2f" % I4_MAX_VAL_TEST_GAP,\n                      int(gap <= I4_MAX_VAL_TEST_GAP), 1,\n                      f"val {fv.summary[\'macro_f1\'][0]:.4f}, test {mf_f:.4f}, chênh {gap:.4f}"))\n    else:\n        items.append(("I4b", "Chênh macro-F1 val/test <= %.2f" % I4_MAX_VAL_TEST_GAP, None, 1,\n                      "chưa chấm được (thiếu --final-val)"))\n\n    # I5\n    if args.latency_p95_ms is None:\n        items.append(("I5", "Cấu hình thời gian thực", None, 2, "chưa chấm được (thiếu --latency-p95-ms)"))\n    else:\n        ok = args.latency_p95_ms <= I5_P95_BUDGET_MS\n        pts = 0 if not ok else (2 if args.latency_method == "proper" else 1)\n        items.append(("I5", "Cấu hình thời gian thực", pts, 2,\n                      f"p95 = {args.latency_p95_ms:.1f} ms (ngân sách {I5_P95_BUDGET_MS:.0f} ms), "\n                      f"đo {\'đúng cách\' if args.latency_method == \'proper\' else \'chưa đủ warmup/synchronize\'}"))\n\n    scored = [i for i in items if i[2] is not None]\n    got = sum(i[2] for i in scored)\n    max_scored = sum(i[3] for i in scored)\n    lines = ["## Tự chấm RUBRIC mục I (đề xuất; giảng viên xác nhận)", "",\n             "| Mã | Tiêu chí | Điểm | Tối đa | Chi tiết |", "|---|---|---|---|---|"]\n    for code, crit, pts, mx, note in items:\n        lines.append(f"| {code} | {crit} | {\'-\' if pts is None else pts} | {mx} | {note} |")\n    lines += ["", f"**Tổng các ý đã chấm: {got} / {max_scored}** (phần I tối đa 20)."]\n    pending = [i[0] for i in items if i[2] is None]\n    if pending:\n        lines.append(f"Chưa chấm được: {\', \'.join(pending)}.")\n    if warnings:\n        lines += ["", "Cảnh báo:"] + [f"- {w}" for w in warnings]\n    lines += ["", "Ngưỡng điểm là TẠM THỜI (xem khối hằng số đầu file eval.py và RUBRIC.md mục I)."]\n    print("\\n".join(lines))\n\n    if args.out:\n        out = Path(args.out)\n        out.mkdir(parents=True, exist_ok=True)\n        (out / "grade_I.json").write_text(json.dumps({\n            "items": [{"code": c, "criterion": cr, "points": p, "max": m, "note": n} for c, cr, p, m, n in items],\n            "total": got, "max_scored": max_scored, "warnings": warnings}, indent=2, ensure_ascii=False))\n    return 0\n\n\n# --------------------------------------------------------------------------- #\n# CLI\n# --------------------------------------------------------------------------- #\ndef build_parser() -> argparse.ArgumentParser:\n    ap = argparse.ArgumentParser(description="Đánh giá Lab Day 2 (DeepWeeds)")\n    sub = ap.add_subparsers(dest="cmd", required=True)\n\n    sc = sub.add_parser("score", help="tính chỉ số cho một nhóm file dự đoán (nhiều seed)")\n    sc.add_argument("--pred", nargs="+", required=True, help="glob của file dự đoán")\n    sc.add_argument("--test-csv", help="labels/test_subset0.csv để đối chiếu tên file và nhãn")\n    sc.add_argument("--labels", help="labels/labels.csv để lấy tên lớp")\n    sc.add_argument("--tag", default="score")\n    sc.add_argument("--out", help="thư mục lưu JSON/CSV")\n    sc.set_defaults(func=cmd_score)\n\n    gr = sub.add_parser("grade", help="tự chấm phần I của RUBRIC")\n    gr.add_argument("--final", nargs="+", required=True, help="glob dự đoán test của cấu hình chung kết")\n    gr.add_argument("--baseline", nargs="+", required=True, help="glob dự đoán test của mốc (T00 + I00)")\n    gr.add_argument("--uncal", nargs="+", help="glob dự đoán test của chung kết khi chưa temperature scaling")\n    gr.add_argument("--final-val", nargs="+", help="glob dự đoán val của chung kết")\n    gr.add_argument("--latency-p95-ms", type=float, help="p95 batch 1 của cấu hình thời gian thực (ms)")\n    gr.add_argument("--latency-method", choices=["proper", "partial"], default="proper",\n                    help="proper = có warmup và synchronize; partial = chưa đủ")\n    gr.add_argument("--test-csv", help="labels/test_subset0.csv")\n    gr.add_argument("--val-csv", help="labels/val_subset0.csv (đối chiếu --final-val)")\n    gr.add_argument("--labels", help="labels/labels.csv")\n    gr.add_argument("--out", help="thư mục lưu grade_I.json")\n    gr.set_defaults(func=cmd_grade)\n    return ap\n\n\ndef main(argv=None) -> int:\n    args = build_parser().parse_args(argv)\n    try:\n        return args.func(args)\n    except (ValueError, FileNotFoundError) as e:\n        print(f"LỖI: {e}", file=sys.stderr)\n        return 2\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'}
for name, content in SOURCES.items():
    (CODE_DIR/name).write_text(content, encoding='utf-8')
sys.path.insert(0, str(CODE_DIR))
for name in ('dataset', 'model', 'losses', 'inference', 'benchmark', 'train', 'experiments', 'eval', 'test_core'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import torch, timm, numpy as np, pandas as pd
print('torch', torch.__version__, 'timm', timm.__version__)
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Implementation:', CODE_DIR)


## 1. Dữ liệu có sẵn và kiểm tra fold 0
Gắn dataset bằng **Add Input** nếu dữ liệu nằm trong dataset Kaggle.
Notebook tự tìm bộ CSV chính thức và ảnh trong `/kaggle/input`, `/kaggle/working/data`.
Nếu có nhiều bản, đặt hai đường dẫn bên dưới. Ảnh cần được giải nén sẵn.


In [ ]:
from dataset import load_split, check_split, CLASS_NAMES
# Điền đường dẫn ở đây khi cần. Không dùng CSV đã tự chia lại.
IMAGES_DIR = None
LABELS_DIR = None
roots = [Path('/kaggle/input'), Path('/kaggle/working/data')]
if LABELS_DIR is None:
    candidates = sorted({p.parent for root in roots if root.exists() for p in root.rglob('train_subset0.csv')
                         if all((p.parent/f'{s}_subset0.csv').exists() for s in ('val', 'test'))})
    if not candidates:
        # The uploaded images.zip contains JPEGs only. Fetch the unchanged official metadata.
        import urllib.request
        labels_path = Path('/kaggle/working/data/labels')
        labels_path.mkdir(parents=True, exist_ok=True)
        label_base = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
        for filename in ('labels.csv', 'train_subset0.csv', 'val_subset0.csv', 'test_subset0.csv'):
            urllib.request.urlretrieve(f'{label_base}/{filename}', labels_path/filename)
        candidates = [labels_path]
    if len(candidates) != 1:
        raise ValueError(f'Found label directories: {candidates}. Attach your dataset and set LABELS_DIR.')
    LABELS_DIR = str(candidates[0])
train_df, val_df, test_df = load_split(LABELS_DIR)
if IMAGES_DIR is None:
    sample = str(train_df.iloc[0].Filename)
    candidates = sorted({p.parent for root in roots if root.exists() for p in root.rglob(sample)})
    if len(candidates) != 1:
        raise ValueError(f'Found image directories: {candidates}. Set IMAGES_DIR to extracted JPEGs.')
    IMAGES_DIR = str(candidates[0])
checks = check_split(train_df, val_df, test_df, IMAGES_DIR)
print('Images:', IMAGES_DIR, 'Labels:', LABELS_DIR)


## 2. EDA và kiểm tra code

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
counts = pd.DataFrame({s: df.Label.value_counts().reindex(range(9), fill_value=0)
                       for s, df in zip(('train', 'val', 'test'), (train_df, val_df, test_df))})
counts.index = CLASS_NAMES
display(counts)
Path('curves').mkdir(exist_ok=True)
ax = counts.plot.bar(figsize=(12,5), title='DeepWeeds fold 0')
ax.set_ylabel('Images'); ax.figure.tight_layout()
ax.figure.savefig('curves/EDA_class_distribution.png', dpi=150)
plt.show()
fig, axs = plt.subplots(9,3, figsize=(9,24))
for k in range(9):
    examples = train_df[train_df.Label == k].sample(3, random_state=0)
    for ax, filename in zip(axs[k], examples.Filename):
        with Image.open(Path(IMAGES_DIR)/filename) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(CLASS_NAMES[k]); ax.axis('off')
fig.tight_layout(); fig.savefig('curves/EDA_samples.png', dpi=100); plt.show()


In [ ]:
# Kiểm tra loss, diện tích CutMix, gộp BN, temperature, optimizer, frozen BN và train/eval.
import unittest, test_core
suite = unittest.defaultTestLoader.loadTestsFromModule(test_core)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()


In [ ]:
from dataclasses import replace
from train import Config, run, run_dir
from dataset import build_transforms, make_loader
from experiments import train_suite, inference_suite, load_checkpoint, final_inference, export_results, restore_config
from inference import predict_logits
from eval import compute_metrics, save_predictions
BASE = Config(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, epochs=12, batch_size=64, seed=0)
print(BASE)
# Kiểm tra augmentation bằng mắt.
from torchvision.utils import make_grid
x, y, filenames = next(iter(make_loader(train_df, IMAGES_DIR, build_transforms(True), 8, True, num_workers=0)))
display_image = x * torch.tensor([.229,.224,.225])[None,:,None,None] + torch.tensor([.485,.456,.406])[None,:,None,None]
plt.figure(figsize=(12,4)); plt.imshow(make_grid(display_image.clamp(0,1), nrow=4).permute(1,2,0)); plt.axis('off'); plt.show()
print('Labels:', y.tolist())


## 3. Kiểm tra pipeline trước khi chạy thật
Bật `RUN_SANITY` để overfit một batch ảnh cố định. Ghi lại loss và ảnh augmentation vào báo cáo.
Loss của logits đồng đều bằng ln(9); loss head ngẫu nhiên chỉ xấp xỉ giá trị này.


In [ ]:
RUN_SANITY = True
if RUN_SANITY:
    from model import build_model
    from train import set_seed
    set_seed(0)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    sanity = build_model('efficientnet_b0', pretrained=False, init='scratch').to(device)
    sanity.train()
    images, labels, _ = next(iter(make_loader(train_df, IMAGES_DIR, build_transforms(False), 8, False, num_workers=0)))
    images, labels = images.to(device), labels.to(device)
    opt = torch.optim.Adam(sanity.parameters(), lr=1e-3)
    print('Uniform loss:', torch.nn.functional.cross_entropy(torch.zeros(8,9,device=device), labels).item())
    losses = []
    for step in range(100):
        opt.zero_grad(); z = sanity(images); loss = torch.nn.functional.cross_entropy(z, labels)
        loss.backward(); opt.step(); losses.append(loss.item())
        if step % 20 == 0: print(step, loss.item())
    plt.plot(losses); plt.xlabel('Step'); plt.ylabel('One-batch loss'); plt.show()
    print('Final batch accuracy:', (sanity(images).argmax(1) == labels).float().mean().item())
    del sanity, opt
    if torch.cuda.is_available(): torch.cuda.empty_cache()


## 4–8. Chạy đầy đủ các thí nghiệm và chung kết
Tự chọn backbone bằng macro-F1 val; chạy 10 ablation và kết hợp các yếu tố thắng.
So sánh 1-view, TTA prob/logit, temperature, fusion và FP16.
Lưu `selection.json` trước test; chung kết và baseline với seed 0/1/2.
Tái sử dụng các lần train giống hệt, ghi rõ nguồn; không suy luận test lại nếu đã có predictions.
Kết quả: `results.xlsx`, `report.md`, `final_mean_std.csv`, `curves/`, `predictions/`, `eval_out/`.
Chỉ khi có `COMPLETE.json` thì tất cả các phần đã hoàn tất.


In [ ]:
assert torch.cuda.is_available(), 'Enable GPU before running the full lab'
from full_lab import run_all
final_results = run_all(BASE, val_df, test_df)
display(final_results)
